In [ ]:
from dataclasses import dataclass
from pathlib import Path
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np

import ipywidgets as widgets
from IPython.display import display, clear_output

@dataclass
class RejectionSamplingData:
    """
    Container for precomputed functions and parameters used in the
    rejection sampling visualization.

    Attributes
    ----------
    z : np.ndarray
        Grid over which densities are evaluated.
    p_prime : np.ndarray
        Unnormalized target density p'(z), here a sum of three Gaussians.
    k : float
        Scaling constant such that k * q(z) ≥ p'(z) for all z in `z`.
    q : np.ndarray
        Proposal/envelope density q(z) evaluated on `z`.
    q_mu : float
        Mean of the proposal density q.
    q_sigma : float
        Standard deviation of the proposal density q.
    mus : np.ndarray
        Means of the three Gaussian components that form p'(z).
    sigmas : np.ndarray
        Standard deviations of the three Gaussian components that form p'(z).
    """
    z: np.ndarray
    p_prime: np.ndarray
    k: float
    q: np.ndarray
    q_mu: float
    q_sigma: float
    mus: np.ndarray
    sigmas: np.ndarray


def normal_pdf(z: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    """
    Evaluate a univariate Normal(μ, σ²) probability density function.

    Parameters
    ----------
    z : np.ndarray
        Points at which to evaluate the density.
    mu : float
        Mean of the normal distribution.
    sigma : float
        Standard deviation of the normal distribution.

    Returns
    -------
    np.ndarray
        The density values at `z`.
    """
    z = z.astype(np.float64)
    mu = np.float64(mu)
    sigma = np.float64(sigma)
    inv = np.float64(1.0) / (sigma * np.sqrt(np.float64(2.0) * np.pi))
    return inv * np.exp(np.float64(-0.5) * ((z - mu) / sigma) ** np.float64(2.0))


def compute_rejection_sampling_data() -> RejectionSamplingData:
    """
    Build the mixture target p'(z), the Gaussian proposal q(z), and the
    rejection constant k, all evaluated on a dense grid.

    The target p'(z) is a sum of three Gaussian components with fixed
    means and standard deviations. The proposal q(z) is a single Gaussian
    with mean at the average of the component means and a widened variance
    to ensure it envelopes p'(z). The constant k is set slightly above
    max_z p'(z)/q(z) for a conservative envelope.

    Returns
    -------
    RejectionSamplingData
        Dataclass with grid `z`, densities `p_prime`, `q`, envelope scale `k`,
        proposal parameters, and component parameters.
    """
    mus = np.array([-2.0, 0.75, 3.5], dtype=np.float64)
    sigmas = np.array([0.6, 0.9, 1.2], dtype=np.float64)
    z_min = float(mus.min() - 6.0 * sigmas.max())
    z_max = float(mus.max() + 6.0 * sigmas.max())
    z = np.linspace(z_min, z_max, 3000, dtype=np.float64)
    components = [normal_pdf(z, mu, sigma) for mu, sigma in zip(mus, sigmas)]
    p_prime = np.sum(components, axis=0).astype(np.float64)
    q_mu = float(np.mean(mus))
    q_sigma = float(2.5 * sigmas.max())
    q = normal_pdf(z, q_mu, q_sigma)
    ratio = p_prime / q
    k = 1.05 * float(np.max(ratio))
    return RejectionSamplingData(z=z, p_prime=p_prime, k=k, q=q, q_mu=q_mu, q_sigma=q_sigma, mus=mus, sigmas=sigmas)


def plot_rejection_sampling_at_z0(data: RejectionSamplingData, z0: float) -> plt.Figure:
    """
    Create a Matplotlib figure illustrating rejection sampling geometry at a given z₀.

    The plot shows the target p'(z), the scaled envelope k·q(z), the rejection
    region between them, component Gaussians, a cloud of samples from q,
    and vertical segments at z₀ illustrating the acceptance probability
    p'(z₀) / (k·q(z₀)).

    Parameters
    ----------
    data : RejectionSamplingData
        Precomputed densities and parameters.
    z0 : float
        Location at which to annotate the acceptance probability.

    Returns
    -------
    matplotlib.pyplot.Figure
        The constructed figure.
    """
    y1 = sum(normal_pdf(np.array([z0]), mu, sigma)[0] for mu, sigma in zip(data.mus, data.sigmas))
    y2 = data.k * normal_pdf(np.array([z0]), data.q_mu, data.q_sigma)[0]
    acc = y1 / y2

    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.plot(data.z, data.p_prime, linewidth=2, label="p'(z) = sum of 3 Gaussians")
    ax.plot(data.z, data.k * data.q, linestyle="--", linewidth=2, label="k · q(z) (envelope)")
    ax.fill_between(data.z, data.p_prime, data.k * data.q, alpha=0.3, color="gray", label="rejection region")

    for mu, sigma in zip(data.mus, data.sigmas):
        ax.plot(data.z, normal_pdf(data.z, mu, sigma), linewidth=1, alpha=0.5)

    rng = np.random.default_rng(0)
    samples = rng.normal(data.q_mu, data.q_sigma, size=200)
    ax.scatter(samples, np.zeros_like(samples), alpha=0.3, color="orange", label="normal sample")

    ax.plot([z0, z0], [0.0, y1], color="blue")
    ax.plot([z0, z0], [y1, y2], color="red")

    ax.set_title(f"Rejection Sampling | z₀ = {z0:.2f} | Acceptance Prob: {acc:.2%}")
    ax.set_xlabel("z")
    ax.set_ylabel("density (unnormalized)")
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig


def plot_rejection_sampling_interactive(data: RejectionSamplingData) -> None:
    """
    Display an interactive ipywidgets UI with a slider for z₀ and a screenshot button.

    On slider movement, the plot updates in-place within an Output widget.
    To avoid duplicate static images in VS Code notebooks, the figure is
    explicitly closed after display. Pressing the screenshot button
    regenerates the figure for the current z₀ and saves it under ./plots/.

    Parameters
    ----------
    data : RejectionSamplingData
        Precomputed densities and parameters.
    """
    slider = widgets.FloatSlider(
        value=0.0, min=-7.5, max=7.5, step=0.01,
        description="z₀", continuous_update=False, readout_format=".2f"
    )
    button = widgets.Button(description="Screenshot", icon="camera", tooltip="Save plot to ./plots/")

    output = widgets.Output()
    state = {"current_z0": slider.value}

    def update_plot(z0: float):
        state["current_z0"] = float(z0)
        with output:
            clear_output(wait=True)
            fig = plot_rejection_sampling_at_z0(data, state["current_z0"])
            display(fig)
            plt.close(fig)

    def on_slider_change(change):
        if change["name"] == "value":
            update_plot(change["new"])

    def on_button_clicked(_):
        z0 = state["current_z0"]
        fig = plot_rejection_sampling_at_z0(data, z0)
        Path("plots").mkdir(exist_ok=True)
        fname = Path("plots").joinpath(f"rejection_sampling/rejection_sampling_{str(round(z0, 3)).replace('.', '_')}.png")
        fig.savefig(fname, dpi=300)
        plt.close(fig)
        with output:
            print(f"Saved screenshot to: {fname}")

    slider.observe(on_slider_change, names="value")
    button.on_click(on_button_clicked)

    controls = widgets.HBox([slider, button])
    display(widgets.VBox([controls, output]))
    with output:
        clear_output(wait=True)
    update_plot(slider.value)


if __name__ == "__main__":
    data = compute_rejection_sampling_data()
    plot_rejection_sampling_interactive(data)